# 🚀 Giai đoạn 4: Ứng dụng Pretrained Transformer (ViSoBERT / PhoBERT)
Fine-tune mô hình Transformer cho phân loại cảm xúc tiếng Việt


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

model_name = "5CD-AI/Vietnamese-Sentiment-visobert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
sentiment_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

sample_text = "Công ty môi trường làm việc chuyên nghiệp, đồng nghiệp thân thiện, sếp tâm lý."
print(sentiment_pipe(sample_text))


> **Yêu cầu môi trường:** notebook này cần `torch` + `transformers` và nên chạy trên máy có GPU (ví dụ Runpod). Môi trường dev hiện tại không cài 2 gói này nên notebook được chuẩn bị đầy đủ code nhưng **chưa được thực thi** — chạy trên GPU rồi cập nhật lại số liệu vào `reports/modeling_hyperparameter_tuning.md`.

## 4.1 Benchmark Zero-shot trên đúng tập Final Test đã khóa của notebook 03
Dùng lại `test_indices` từ `models/train_test_features.joblib` để lấy đúng các dòng final test mà notebook 03 dùng, đảm bảo so sánh ViSoBERT với các mô hình ML là công bằng (cùng tập test, cùng nhãn thật). Văn bản đưa vào ViSoBERT dùng cột `clean_basic_text` (giữ nguyên cấu trúc câu tự nhiên) thay vì `clean_advance_text` (đã tách từ ghép, chỉ tối ưu cho ML cổ điển) — theo đúng thiết kế 2 tầng tiền xử lý trong README.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

from src.features import load_feature_split

artifact = load_feature_split(PROJECT_ROOT / 'models' / 'train_test_features.joblib')
test_indices = artifact['test_indices']
y_test = artifact['y_test']

reviews = pd.read_excel(PROJECT_ROOT / 'data' / 'processed' / 'reviews_cleaned.xlsx')
test_texts = reviews.loc[test_indices, 'clean_basic_text'].fillna('').astype(str)
print(f'So mau final test: {len(test_texts)}')

In [ ]:
# Nhan tho ma pipeline tra ve tuy theo checkpoint (vd 'POS'/'NEU'/'NEG' hoac
# 'LABEL_0'/'LABEL_1'/'LABEL_2'). Chay cell nay tren GPU truoc de biet chinh xac
# nhan tho la gi, roi chinh sua RAW_LABEL_TO_SENTIMENT cho khop neu can.
RAW_LABEL_TO_SENTIMENT = {
    'POS': 'Positive', 'NEU': 'Neutral', 'NEG': 'Negative',
    'POSITIVE': 'Positive', 'NEUTRAL': 'Neutral', 'NEGATIVE': 'Negative',
    'LABEL_0': 'Negative', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positive',
}

batch_size = 32
raw_predictions = []
for start in range(0, len(test_texts), batch_size):
    batch = test_texts.iloc[start:start + batch_size].tolist()
    outputs = sentiment_pipe(batch, truncation=True, max_length=256)
    raw_predictions.extend(item['label'] for item in outputs)

unseen_labels = set(raw_predictions) - set(RAW_LABEL_TO_SENTIMENT)
if unseen_labels:
    raise ValueError(
        f'Nhan {unseen_labels} chua co trong RAW_LABEL_TO_SENTIMENT, '
        'hay bo sung anh xa roi chay lai.'
    )

y_pred_visobert = [RAW_LABEL_TO_SENTIMENT[label] for label in raw_predictions]

## 4.2 Đánh giá & so sánh với các mô hình ML (notebook 03)

In [ ]:
visobert_f1_macro = f1_score(y_test, y_pred_visobert, average='macro')
visobert_accuracy = accuracy_score(y_test, y_pred_visobert)

print(f'ViSoBERT (zero-shot) - Accuracy: {visobert_accuracy:.4f}; Macro F1: {visobert_f1_macro:.4f}')
print(classification_report(y_test, y_pred_visobert, digits=4))